# Thesis figures: restyle and re-render

**Runtime: CPU is fine.** No GPU needed. Takes about a minute.

Every figure in the thesis is drawn by the scripts in this bundle, and every value they plot is
read from a stored result file. Nothing is typed in. That means you can restyle freely without any
risk of changing a number: if you edit only the palette, the data is untouched by construction.

## How to use this

1. Run cell 1 to mount Drive and set up.
2. **Edit cell 2, the palette.** That is the only cell you need to touch.
3. Run cell 3 to render, cell 4 to preview them inline.
4. Run cell 5 to save back to Drive. Then tell me and I will rebuild the thesis with them.

If you want to change more than colour, cell 6 opens any individual script for editing.

**No credentialed data is in this bundle.** The result files carry aggregates only; they were
filtered for clinical text and identifiers before being copied.

### 1. Setup

In [ ]:
!pip -q install matplotlib numpy --upgrade 2>/dev/null

import os, shutil, pathlib, json, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

SRC = '/content/drive/MyDrive/This is paper/02_figures/colab_bundle'
WORK = '/content/figures'
shutil.rmtree(WORK, ignore_errors=True)
shutil.copytree(SRC, WORK)

# The scripts were written to run from the repository, so they carry an absolute
# path to it. Point that at the copied bundle instead.
REPO = '/content/repo'
os.makedirs(REPO, exist_ok=True)
if os.path.exists(f'{REPO}/results'):
    shutil.rmtree(f'{REPO}/results')
shutil.copytree(f'{WORK}/results', f'{REPO}/results')

for p in pathlib.Path(WORK).glob('*.py'):
    t = p.read_text()
    t = t.replace('/Users/rickygole/provenance-encoder', REPO)
    p.write_text(t)

os.chdir(WORK)
print('scripts :', len(list(pathlib.Path(WORK).glob('*.py'))))
print('data    :', sum(1 for _ in pathlib.Path(f'{REPO}/results').rglob('*.json')), 'json files')
print('ready')

### 2. The palette. This is the cell to edit.

Four colours do all the work. They are used across every figure, so changing them here changes the
whole set consistently.

One thing worth keeping if you can: the four differ in **lightness** as well as hue, so the figures
still read correctly if the thesis is printed or photocopied in black and white. The printed
lightness values are shown when you run the cell. If you pick four colours of similar lightness the
figures will look fine on screen and turn into mush on paper.

Some palettes to try, or write your own:

```
# current, muted navy
DARK, MID, FILL, DEAD = "#1f3a5f", "#7f8c9b", "#cdd7e0", "#b0b6bd"

# warmer, higher contrast
DARK, MID, FILL, DEAD = "#7f2704", "#d94801", "#fdd0a2", "#bdbdbd"

# teal
DARK, MID, FILL, DEAD = "#00441b", "#41ab5d", "#c7e9c0", "#bdbdbd"

# blue to orange, colour-blind safe
DARK, MID, FILL, DEAD = "#08306b", "#4292c6", "#fdd0a2", "#bdbdbd"
```

In [ ]:
# ---- EDIT THESE FOUR ----------------------------------------------------
DARK = "#1f3a5f"   # bars, lines, emphasis
MID  = "#7f8c9b"   # secondary series, reference lines
FILL = "#cdd7e0"   # bar fills, shaded bands
DEAD = "#b0b6bd"   # de-emphasised or retracted elements
# -------------------------------------------------------------------------

ACCENT_RED = "#b03030"   # used sparingly to mark a failure or a retraction

import pathlib, re
def luminance(h):
    r, g, b = [int(h[i:i+2], 16) for i in (1, 3, 5)]
    return 0.299*r + 0.587*g + 0.114*b

print("greyscale lightness, should be well separated:")
for n, h in (("DARK", DARK), ("MID", MID), ("FILL", FILL), ("DEAD", DEAD)):
    print(f"  {n:5s} {h}  {luminance(h):5.1f} / 255")
gaps = sorted(luminance(h) for h in (DARK, MID, FILL))
print(f"  smallest gap between the three main tones: {min(b-a for a,b in zip(gaps, gaps[1:])):.0f}")
print("  (under about 40 they start to merge in a black and white print)")

# Push the palette into every script.
for p in pathlib.Path('/content/figures').glob('*.py'):
    t = p.read_text()
    t = re.sub(r'GREY_DARK\s*=\s*"#[0-9a-fA-F]{6}"',  f'GREY_DARK = "{DARK}"', t)
    t = re.sub(r'GREY_LIGHT\s*=\s*"#[0-9a-fA-F]{6}"', f'GREY_LIGHT = "{MID}"', t)
    t = re.sub(r'GREY_FILL\s*=\s*"#[0-9a-fA-F]{6}"',  f'GREY_FILL = "{FILL}"', t)
    t = re.sub(r'ACCENT,\s*MUTED,\s*LIGHT,\s*DEAD\s*=\s*"[^"]*",\s*"[^"]*",\s*"[^"]*",\s*"[^"]*"',
               f'ACCENT, MUTED, LIGHT, DEAD = "{DARK}", "{MID}", "{FILL}", "{DEAD}"', t)
    p.write_text(t)
print("\npalette applied to every script")

### 3. Render

In [ ]:
import subprocess, pathlib, os, sys, time
os.chdir('/content/figures')

# Every figure the thesis places, and the script that draws it. Two scripts are
# needed: make_figures.py deliberately skips three figures whose data it could
# not trace when it was written, and make_figures_extra.py draws two of those.
USED = {
 "fig-pipeline-status":    "make_figures.py",
 "fig-intro-headline":     "make_figures.py",
 "fig-rule-precision":     "make_figures.py",
 "fig-exemplar":           "make_figures.py",
 "fig-disc-strata":        "make_figures.py",
 "fig-disc-actionability": "make_figures.py",
 "fig-met-exchange":       "make_figures.py",
 "fig-prevalence-response":"fig_batch2.py",
 "fig-scaling":            "fig_batch2.py",
 "fig-power":              "fig_batch2.py",
 "fig-forest":             "fig_batch2.py",
 "fig-threshold-surface":  "fig_threshold_surface.py",
 "fig-falsification":      "make_figures_extra.py",
}

for f in pathlib.Path('.').glob('fig-*.pdf'):
    f.unlink()                      # start clean so nothing stale survives

SCRIPTS = ["make_figures.py", "fig_batch2.py", "fig_threshold_surface.py",
           "make_figures_extra.py", "fig_spec_curve.py", "fig_gap_trajectory.py"]
for s in SCRIPTS:
    if not pathlib.Path(s).exists():
        print(f"  absent  {s}"); continue
    r = subprocess.run([sys.executable, s], capture_output=True, text=True)
    print(f"  {'ok  ' if r.returncode == 0 else 'FAIL'} {s}")
    if r.returncode != 0:
        print("       " + r.stderr.strip().splitlines()[-1][:160])

print("\n--- every figure the thesis uses ---")
missing = []
for name, script in sorted(USED.items()):
    if pathlib.Path(f"{name}.pdf").exists():
        print(f"  ok       {name}")
    else:
        print(f"  MISSING  {name}   (from {script})")
        missing.append(name)

extra = sorted(p.stem for p in pathlib.Path('.').glob('fig-*.pdf') if p.stem not in USED)
if extra:
    print("\nalso rendered, not placed in any chapter:")
    for e in extra: print(f"  {e}")

print()
if missing:
    print(f"{len(missing)} of {len(USED)} MISSING. Do not save; tell Claude which ones.")
else:
    print(f"all {len(USED)} figures used by the thesis rendered successfully")

### 4. Preview

These are the thirteen the chapters actually use. The other seven are rendered but unplaced.

In [ ]:
!pip -q install pdf2image 2>/dev/null
!apt-get -qq install poppler-utils 2>/dev/null

from pdf2image import convert_from_path
import matplotlib.pyplot as plt, pathlib

# Preview everything that rendered, the thirteen the thesis places first.
placed = sorted(USED)
others = sorted(p.stem for p in pathlib.Path('.').glob('fig-*.pdf') if p.stem not in USED)

for group, names in (("USED IN THE THESIS", placed), ("RENDERED BUT NOT PLACED", others)):
    if not names: continue
    print("=" * 70); print(group); print("=" * 70)
    for name in names:
        p = pathlib.Path(f"{name}.pdf")
        if not p.exists():
            print(f"  missing: {name}"); continue
        img = convert_from_path(str(p), dpi=110)[0]
        plt.figure(figsize=(11, 11 * img.height / img.width))
        plt.imshow(img); plt.axis("off")
        plt.title(name, fontsize=10, loc="left")
        plt.show()

### 5. Save back to Drive

In [ ]:
import shutil, pathlib, datetime
OUT = '/content/drive/MyDrive/This is paper/02_figures'
n = 0
for f in pathlib.Path('/content/figures').glob('fig-*.pdf'):
    shutil.copy(f, f'{OUT}/{f.name}'); n += 1
print(f"saved {n} figures to {OUT}")
print("tell Claude they are updated and the thesis will be rebuilt with them")

### 6. Optional: change more than the colours

Each figure is drawn by a function in one of the scripts. To change sizes, labels, fonts or layout,
print the script, edit it, and re-run cells 3 to 5.

`make_figures.py` draws the pipeline status, headline, rule precision, exemplar, funnel, prevalence,
tracking, strata, actionability, exchange and sharpened figures. `fig_batch2.py` draws prevalence
response, scaling, forest and power. The remaining scripts are one figure each.

**One rule if you edit beyond colour:** these scripts read every number from the result files in
`/content/repo/results`. Do not replace a computed value with a typed one. If a figure should show
a different number, the number is wrong in the data, not in the figure.

In [ ]:
# Print a script to read or edit it. Change the filename as needed.
print(open('/content/figures/make_figures.py').read()[:4000])